# P51 — SWE-bench: ¿pueden los modelos resolver incidencias reales de GitHub?

## 1. Título y paper

**Paper:** *SWE-bench: Can Language Models Resolve Real-World GitHub Issues?*  
**Autoría:** Carlos E. Jimenez, John Yang, Alexander Wettig, y otros  
**Año y venue:** 2023 · arXiv:2310.06770 · ICLR 2024  
**Nivel:** L3 · **Motor:** `swebench`  
**Ficha completa:** [`P51_swebench`](../../papers/foundational/P51_swebench/README.md)

**Hito:** Cambia el criterio de evaluación: no si el código parece bien, sino si los tests del repositorio real pasan.

- [arXiv:2310.06770](https://arxiv.org/abs/2310.06770)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Los benchmarks de programación usaban problemas de juguete autocontenidos y se saturaban rápido; no medían nada parecido al trabajo real de mantener un repositorio.
2. Ejecutar una implementación mínima de la propuesta: Construir el conjunto a partir de incidencias y parches reales de proyectos populares, y evaluar con un criterio objetivo: aplicar el parche generado y ejecutar los tests del propio repositorio.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P13
- P28


## 4. Intuición

La pregunta no es si el código parece correcto. Es si, aplicado al repositorio real, los tests que ya existían pasan. Un test no se deja convencer.


## 5. Concepto mínimo

```text
Benchmarks previos:  problema autocontenido → ¿la salida coincide?
SWE-bench        :  incidencia REAL de un repo real
                     → aplicar el parche generado
                     → ejecutar los tests DEL PROPIO repositorio
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('swebench', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Qué proporción «parece correcta» en el ejemplo? ¿Y cuál pasa los tests?
2. ¿Por qué es tan grande la diferencia?
3. ¿Basta con que compile?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('swebench', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('swebench', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Medido por apariencia el sistema resuelve mucho más que medido por tests. Esa brecha es el problema entero: **los criterios blandos inflan**, y en programación es especialmente fácil que algo parezca correcto y no lo sea.


## 10. Comentario pedagógico

El propio benchmark tiene una debilidad conocida: las incidencias son públicas y anteriores al corte de datos de muchos modelos, así que hay riesgo de contaminación. Existen variantes verificadas y filtradas justamente por eso.


## 11. Error o anti-patrón deliberado

Anti-patrón: reportar «resuelve el 60 %» sin decir con qué criterio.


In [ ]:
criterios = {'parece correcto': 'juicio humano rapido o de otro modelo',
             'compila': 'necesario, muy lejos de suficiente',
             'tests pasan': 'el criterio del benchmark',
             'revision humana acepta': 'el criterio del mundo real, aun mas duro'}
for k, v in criterios.items():
    print(f'{k:<24} → {v}')

## 12. Corrección

Un reporte creíble nombra el criterio y las condiciones:


In [ ]:
reporte = {'criterio': 'tests del repositorio pasan',
           'conjunto': 'que subconjunto y de que fecha',
           'contaminacion': 'si se comprobo solapamiento con el corpus',
           'coste': 'llamadas al modelo e intentos por incidencia',
           'andamiaje': 'que agente/herramientas, no solo que modelo'}
show(reporte)

## 13. Desafío guiado

Calcula la tasa con cada criterio y ordénalos de más blando a más duro.


In [ ]:
r = run_paper_lab('swebench', seed=3)['result']
show(r)

## 14. Desafío autónomo

Toma cinco incidencias cerradas de un repositorio propio, pide a un modelo que las resuelva y evalúa con los tests reales. Compara con tu impresión al leer el parche: mide tu propia brecha.


## 15. Evidencia de aprendizaje

Guarda la tabla de tasas por criterio y tu formato de reporte con criterio, contaminación y coste.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P51_swebench/README.md) · evaluación formal: [`assessments/papers/P51_swebench.md`](../../assessments/papers/P51_swebench.md)


## 16. Cierre

Ya se puede medir capacidad con un criterio duro. Queda mirar dentro del modelo.


## 17. Conexión con el siguiente hito

- evaluación de agentes
- P16

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
